In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset



In [3]:
class LSTM_DataSet(Dataset):
    def __init__(self, df, window_size, features, targets, wait_id=0):
        self.df = df
        self.window_size = window_size
        self.features = features      # list of feature column names
        self.targets = targets        # list like ["action_id", "pos_x", "pos_y"]
        self.wait_id = wait_id

        # Nested:
        self.X_per_match = []
        self.Y_per_match = []

        # Flattened:
        self.X_flat = []   # each: [T, num_features]
        self.action_flat = []  # each: scalar int
        self.pos_flat = []     # each: [2]
        self.pos_mask_flat = []  # each: bool (should we include in pos loss?)

    def get_match_ids(self):
        return self.df["match_id"].unique().tolist()

    def transform(self):
        match_ids = self.get_match_ids()
        self.X_per_match = []
        self.Y_per_match = []

        for mid in match_ids:
            match_set = self.df[self.df["match_id"] == mid]
            match_X = []
            match_Y = []

            for i in range(len(match_set)):
                start = max(0, i - self.window_size)

                if i == 0:
                    window_frames = []
                else:
                    window_frames = match_set.iloc[start:i][self.features].values.tolist()

                if len(window_frames) < self.window_size:
                    pad_rows = self.window_size - len(window_frames)
                    padding = [[0] * len(self.features)] * pad_rows
                    window_frames = padding + window_frames

                target_row = match_set.iloc[i][self.targets]
                target_list = target_row.values.tolist()  # [action_id, pos_x, pos_y]

                match_X.append(window_frames)
                match_Y.append(target_list)

            self.X_per_match.append(match_X)
            self.Y_per_match.append(match_Y)

        # Now flatten across matches
        self.flatten()

    def flatten(self):
        self.X_flat = []
        self.action_flat = []
        self.pos_flat = []
        self.pos_mask_flat = []

        for match_X, match_Y in zip(self.X_per_match, self.Y_per_match):
            for window_frames, target_list in zip(match_X, match_Y):
                action_id = int(target_list[0])
                pos_x = float(target_list[1])
                pos_y = float(target_list[2])

                self.X_flat.append(window_frames)        # [T, num_features]
                self.action_flat.append(action_id)       # scalar
                self.pos_flat.append([pos_x, pos_y])     # [2]

                # mask: true if we should use this in position loss
                # i.e., not wait and position is valid
                use_pos = (action_id != self.wait_id) and (pos_x != -1) and (pos_y != -1)
                self.pos_mask_flat.append(use_pos)

    def __len__(self):
        return len(self.X_flat)

    def __getitem__(self, idx):
        x = torch.tensor(self.X_flat[idx], dtype=torch.float32)          # [T, F]
        action = torch.tensor(self.action_flat[idx], dtype=torch.long)   # scalar
        pos = torch.tensor(self.pos_flat[idx], dtype=torch.float32)      # [2]
        pos_mask = torch.tensor(self.pos_mask_flat[idx], dtype=torch.bool)  # scalar

        return x, action, pos, pos_mask

In [4]:
full_df = pd.read_csv('full_cleaned_dataset_lstm.csv')
features = [col for col in full_df.columns if col not in ['match_id',"id" ,'action', 'pos_x', 'pos_y']]
targets = ['action', 'pos_x', 'pos_y']

In [5]:
dataset = LSTM_DataSet(full_df, window_size=3, features=features, targets=targets)
dataset.transform()

In [6]:
train_transformed = LSTM_DataSet(full_df, window_size=10, features=features, targets=targets)
train_transformed.transform()
test_transformed = LSTM_DataSet(full_df, window_size=10, features=features, targets=targets)
test_transformed.transform()

In [12]:
#building the lstm class with 2 heads, one for action and one for positions
class LSTM_BehaviorCloning(nn.Module)
    def __init__(self):
        super(LSTM_BehaviorCloning, self).__init__()

[False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 True,
 False,
 True,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 True,
 False,
 False,
 

Example input shape: torch.Size([10, 205])


AttributeError: 'tuple' object has no attribute 'shape'